# 🎬 Flock Clip Engine — runs in your browser

Turns one long video into captioned vertical clips. **Nothing gets installed on your computer** — this all runs on Google's machines.

**How to use this page (once, top to bottom):**
1. *(Optional but faster)* Menu bar → **Runtime → Change runtime type → T4 GPU → Save**
2. Click the **▶ play button** on the left edge of each gray box below, **in order**. Wait for each to finish (the spinner stops) before the next.
3. Box 3 is where you paste your video link and API key.

At the end, a zip of your clips downloads to your computer.

### 1️⃣ Install the tools (2–3 min). Click ▶ and wait.

In [ ]:
!pip -q install anthropic yt-dlp pyyaml faster-whisper
!pip -q install mediapipe opencv-python-headless || echo "(no face tracking — will center-crop)"
print("\n✅ Install done. Go to step 2.")

### 2️⃣ Load the engine. Click ▶ (instant).

In [ ]:
import base64, io, tarfile
BLOB_PARTS = [
    "H4sIAL9HS2oC/+196XYbR5amf+MpoqDTLcAGkgAILoIL7qYkStZYCw8pl6oOjIGTiQCRZiITnZkgCbNZp3/NA8yZJ5gf82D9JHO/",
    "eyNyAUAtPbamqoSssoiMjLix37h7ODvOzr+euDffa3es469+l6clz31/W63dbv4b6e1Wp935St189RmeRZK6MVX/1Zf5dA7VLPVn",
    "ut8+ONztPHrUfbTntNrtg/Z+u/LV9vmHf7zAn490eOGHemc08kM/HY2c+fI33//73e69+3+v1f2qvdfpdPYO9ve7B7T/u51O6yvV",
    "+pz7P46i9H35PvT97/SpVqtPaAmoY14C6j//43+pRAeT5jRKUj1Wb+aLxJnHUTNJl4FWVzpOfc8Nmlg2KtYhnRk6diqVs9S90ElP",
    "+SH9SVXzO5XGbph4sX+u8XYVee45fkz8cDyaRTMdpgnevQXnjvUkdmec1XPnqR+FiSSjBqdy7HpTNYvGC2qDn1AtYz3HpzANlipe",
    "hKF7Tl/ccKyuYz/VyJHqeKbHvptSOjV64npUYRqp6yi+3PnjL9H5dzsqiajMspKg8VRvqKixfpjMtYe+RzHVj9VBL06FBqpSGY1o",
    "BBJq3Wik+qractpOq/p3jSedv4nzf3f9/G9vz//Pcv4fFM7/3VbrsNN1Dvdajw4Ptsf/l3b+L1I/+K3P/o84/7tEAMj5v99q7+62",
    "af/vdrH/t+f/Zzn/z6ZuTMfdVAdzOtx6yovCiX+hgsgd03HeUJPJbK4vdiYTogPogLyO3TkyNlQSLC4SBwfjJI5majSaLNJFrOls",
    "9GfzKE7pcA2j1OXTvFIxab8kUWh/x9r+SqZYfNnb4pzq8nSSCOS5m04D/9yCPaHXDN7SnQWVypM3r5+9eD46OXr7PR3MyFCj5vgB",
    "Nabu0CkeBVe6Vnfm1NMwNX/UjqpKX6v4eU4Ey9gBODroK2M94REYSY4amtBjwOrfVZLGVEuhzjqIlbHvpb2Koufap2wRUShcrK7c",
    "RE3kC55Y0yiF3HAncSd6hHpqk7qpFaPqT5a1VN+kPVTVUDP3JtAhqKuU6t1vcXX0RWAmlBZrhwatFlcH//2n65+S5rDaUFX6D0Cc",
    "ILrWca1ed6iIP6/VN5T6KRk1h9+gUJP+SSSLaWgy6EkDhgYA5amDOqoCedjBIiqs5s3GPRX4STqgjMOG+vrry2tpazahzpNoNg80",
    "kVQnkiB9oEV0ugiVW8jZULHrJ7QCZThBFi6wTpOUCJVYRaGauH5ASbwCAQQFqVeFykyjGrb0KFqk80XafxsvtAyO+cktZSD+hOE4",
    "0nkvGmv1h75qFaaPWqUVtRbH5nEcR3Et+4ZnQotqNgMtivZRg2u3KwDv6j11+1A9dH6JfG5f/e6nUDJJ5wbNDuHF3vCumkEuzQiy",
    "mmHnPTkaU1NC0KVJ7cof60hWKg99uqDhHtDKaWD5DPPhFlC1a3+cThu0+/2LaUrTOlHpVBOZHhMZz7Cw0LQ7IyIfJU+IUqfNrww2",
    "+JY6GQSJOne9S1DXtLN4yn4WpKGa/s92wq6nOrTFiIgPH6agtVMqrscMupYQW2CwjTpf+ME4IbxAfAbmnyZO+Wndsc23cyWIw7me",
    "+t60VjXgq/V8usyiwEoYlOYpy4xFf4V/NSaTX4kFIh5gJD1PkHTVa1Ub5fLNZBpd09lJe0JzHsnelxGVAWVo0QR/gPiwt9JY5qie",
    "gxvWs58CgxqM7A4wQ1KzK4PGoD6o2kYNB63hKk6hGa7J90GVm1Ed1hulVNMsqpHLPlDPaAIwez078vPYB29mGuKHkwgTayYRy1rf",
    "+PQ9jMLmrzqOVC2MlGwrBYRrpmjzXhxUpRIeFr80GsMPbVIZo5lBW9qNacLj6p94uTtf/0vtp/Ftp7F3V7/JfhH8wqbKtjedSWr2",
    "3v2MLbwIxpxzrMFJgjfONxl2yS03+65a2pgY6ZlzEUeLea1dN0NvEzp1i+KZSR1hhgvnSkNF57/wln0dhVrah6+O5MY41HhRjBez",
    "eVKjzA1mhMO032koahmGzk083+/TnCY6q43mcbxamdkgptmFtcZHJ59aDpfjagnUllL+x3y2/P+W/y/K//dah87Bo0cH7S3//8Xx",
    "/yK8/e0lAB/g/9v7B/u5/H9/D/z/Af3Z8v+fif9n8XebZf8vZAmo55q4d2KAPTdQSbSIPe3M5l3F3Lir/hIt3i6Iiv/x9CX4QJsR",
    "5N+nigMM438/m89fHGSy6URIGspGFmxNGmh4Zcj3C+wPfqzyPKgHNG25g8LtOOppdB0yLQRa8Wf59jO0Di76mzGbqMeZXY79uCbi",
    "hMQQrEQeJ+kouiwQrWBe+lwCgoZ8PKuVjI2RNNY2JOB3arXqNE3nvZ0d0Mr4meB3vcDZPFCPoWt5ddJVizn6024dtuY8jcTyhelS",
    "TWhyrMpGeXE0J6b6Uut5wlNFNL4baicnhNdZpGXaHAfzNcaH+ZlzqpyHbCA8xR/7qH84AL1OfRt+gwzuYuxHktR1hztIyjLw2xrs",
    "5kzHF7opPECTejBzmYvCcK3mjQwHAcao/E3Gc43D0kQX5wOYxJ4VFkn+uqNv5sTiLBLIS7J8hmeg7A5PblIrzELOQjyjMX0dpc+i",
    "RTgWPoJKlKAAQiaMgliBGp4nlGEaxtaL5kvAaSBvideg99/giHa29h9b+49c/9Npd/edVvfwUbezvyUAvzD6L1fZ/7Y04Pvpv/Zu",
    "d//A6n+6rVYH9h/79GdL/31O+q/DhMPb3GqDpe7v3pw+bb48/tPxSwUsQWM1mydOpfIuiseFFEUUkMiNcfgxiWclydfTKNBKVliP",
    "iCPXS9noYx5BzEj0iBu70aXOzD4aFcgYk7l2L3XcdK8BWaxDIFd2g0CJ7QcUAJdhdM0KAoiWNWxErtEuH5SNP6ZmPna9S8rLSoYm",
    "0RDBTVVO1//8H/9TvXr5Z/Vu6lNVRDY+P/mx6XqeDnTssvFHqI7mREapMz/wPXqrvdKpG9QdIboi6CpeuZ4jgK8FDEEH4IIBTUOF",
    "WhMp+eTHp0dMjlE2Pf4WtFYCIE9OfmSihHpA/az5oRc4gFs3gMdErV1Q3wUwGjmJtSYqcu4um1Rjk0ZlkWp1dPJC1RIIbAM9Sbmm",
    "ZbQAUUiUHrW6UnkjItrEm+qZq2oYqMSB4M/QHbdVTqr21EB+0q/qxKVFMFkEoU6MhJvoU/rQ7ji7XUjMw7G8PTrEV5k0FDw7OT76",
    "4fh01GpV7xrKcZzhXeUTOYMo+SSuIJepGuYgx2ZFnUiRQWiw2oL6wKyD+ncWvBJFiD9lchKC52isg55R/VUDF0TqVYcGZey7sf8r",
    "re3zKAroG5P9ZdbjgXqqJ+4iSHu86O5ZW5jeeoOXg12Xfxbq3DSTgNtfNL9R4lzoVIdXteqTly9ORsevn794fTw6OjvFTLmLNKpm",
    "Iu8MQN98yWnNHLZsD8o98pORiwaOEmkgEavcqnydVzbwNfmSqm6qF9DzajkzFR5RssxPQ4a4z/9acr0Aw6dxogVJu4p1NPJrZFpU",
    "1PhkoMtZPq4WtDTr5QaY9tsGaNlS6Ju/94DP9vQG8PabUYpsYFqE1/iTGywyZcUiBCIMiyZ32cK+Nb+soqKgeqDZaxS2Pf+9W2Mx",
    "ZDNtHEu7i/KtweueFcBQhufKRindNKX5pAnAMxJ6JDacfoYwCNyRcZPjg7B80hB8XjhnoBaE3tBoI59H0Zhy8t5ipPckCtzznXd+",
    "OI6uk52XhBxvMvWhqwiTBdnWstpHh7ZheEG70aCvsoqREU258xbjGECv0Htpzlhf+R4wSNVbjF3ZS1M3GeEt20PefCGgvWg2B+6m",
    "7JMgctP2PpewQDIoUoxOy0Mpl9DSokK3ORLqlRCS/b1bSN9lFlqP/cUMqebXHTBIzSxem2g0g1C0FXtYQ7UN07q+/GnYPozS5Vz3",
    "zYuxL9AXbOTZUDCVnDkFdFxQgfIEj/IJNjKUK3cM6w0adCNFqWS7pFdYXwR4IDpQPlr1BZCErTjfMvh4zfiDvjmy1yhpMFxhuQWB",
    "wbolHNduVw8AZU/FawZhjSka6/nsKRmDGKrx3NauRbZDPd7dVEQO0nIBSrsv++aTtpyxvJW5b3YzZzj343dwgVji7ftBeunKJwIJ",
    "x5rZqq8jVdzetaS06ZSbJAtagPVvebZmtJ990DeEp4hSHHu0ARMlRBLoGhBVsspMcdr1tE0aapFoQ3UKms4QL1Oz3z8bvX3zw/Fr",
    "3vraHa9YE8i2psGxO93qyF+5c0IEPsEJljJWKnRpyYK+wsDA4GMR+umSRnseJd8qu+tAiqI5/7YgJEefDapyzMzMI+zjbNZKG5qa",
    "0czg7pgGNW2OJk7TxlrJ3Q+W3F0tWUAKG8vJ90KpFaTx4crM7o11AizdLw5wESsUTSAy9ADibxTFo+lkhOHq4597kEYlN5P5OEQh",
    "DeLeVC3SIDRZQgsZ6gDm4JxyXDbWsQeh7msDjPd/HZOfq/KLD+2PlI4m/V/CPgP5NfxUBDQwycNPwEEDThz+TmhoA5lJyAdkdK+4",
    "IeeBm0IOXLKAMmlOsqSdPKOSOC+fujHxg1W2UslyzIgvJN7TZHHj2X7X2qytUHPr2HCFut+IHC3/YTEO4QtJIqwYuL/6hDKSiLFA",
    "zJyjsMXCEAsfaAmUNIqJhc2Noj4TTcEF3QR8je0EG4OMeBzsTv/wic+y/jUgnFo48utlZED1FpEA5wZflnrTESiOfns/N1Wi8aFB",
    "zehF8RAhxgmrqhnoK8LLOVaQNiHvyHRhRsfRevvyHLXADS8W7oUewUqvL20cVG1qdbhC/qx0JYPLIGu2eIZaqPhacxrK9NgM79oe",
    "K6kZeO2PvKkbj7JBSIy1T6V4HoKXuL2zXJhZw7wtClyjPQ2LvBNlLfbkaX5in/hz4mRoH9ERS5NKaDmN6GTtbwS4caAM/IwE6ONN",
    "ZrxeMGRbG80kwaAxzrfdqxXgNEyZTyIS1yfnv4rzDVZlDRGVue5hpS5Cnh9arDxIiarFbqzr4l516c+/gBPBnoUmoVE6IuoffUaU",
    "GeKCYncDjfo6ahJZSpuM+IaQ1saVdqCUC3heQP8B8T41ANXZ0x9gu3jlNjsZml9jOOuGen0JqZqb0BZS+mZOp5WfKha3GdTuTSMg",
    "2CgXO7E0MyM/Q0JORKXSWLBvGhey2CM7E+Zmi/EyQYFlOoV4k3jZ0JhjRiEc3jS007m4pUzGioDgdZS+gKE1Vrcer5gpV7NByNpH",
    "q/fah3n1UqeO+jHJRAj9h3YnPmyAa/ItVAVqnE7+HKqdKwdzo0NnHlMOj5qpDf2dzGi5jUSn+40Z8m8WKU2XG3o6YQQ1I1rbTgJ1",
    "PnGqhrAzC6Jw9pWIhTRe5nvTnMl8oK7ayHKiAxAOCJAr1w/gPWj0vfrG0/NUHfMfmsE1s33Gtv84qrGt/ner/y3Z/+0eOq3uwUH7",
    "8HCr//3C9L/spP07OAB+wP5v92Av9/8/6ED/u7u3d7DV/35W/e8uUx3PoPDbmcfReOExAcNrYhG48ZJo1JiOc6RCAWxkgTMXYrzE",
    "yMPYAz+K2WsHbhNE+FaPEnc+Bf33LIi8S/XWDS7pDT75RPlwsbqj3sHl6AZUEFxOKizCAxFgyJ7Hx8/enB7noQHOxdmCm4y8Rph4",
    "7l9w+AErbrumLBEIL4QxaCh2ElJEQFTCSBE56VKKmwJAoo0Khqh0Rz2NiXQkQo9I8lgTvXW+ND6RO7mPYBbUYETtXkAn/onqzVh/",
    "mtGjdeJorGs6IURZjrg1otflj1aaIU3vsXviil5y7DJDnDuI5KWFGpK+URYBIiR9sds0k7d37AyYs5ugJeXrGvGUw7c8/Rk6RwM8",
    "n8YuzUJAqIgIxbFOPJDWIgfmlUAQNZbThFdRilVUzyfYLAChiUU3GSM2BfSaCctiamaeaOxmCfFI6lIv+4E7Ox+76vKqp5pUde3y",
    "atAiNooIfDi9FISJYFZpsAaGFRR+Er4xI8PgEZOZ8WE5B8m5h5WM+7yOI3jUxtB94btpZD5O4IrREM6YNaRkiQgpTrvMhpbrW5c3",
    "QmYUEtVdy5rIwjCThqrqvY2Sh7xLlJvI+UTHV3rk0UzloEx38kaW9ZVGgp5NZE8lAe095WLyiLNhnBFrXtSyl+duKGaq4H0gTS6x",
    "A2X+m9rVKvPMUx8xQtQfzThiQGjVq3C9g6b+vqqqqrhiyoDQLsGA8gQVpnjg93z1jQoLznpF0a8B9jGjGut54HrCRPVl9OxUb8yP",
    "1vwCMcccTdLhYsZKl1oBzqAXDu+pLVvDAzT/l2FhSgnAxiKUsa/CtU/rE1sq0C59+rTNUVBL5ziiwXtuXTZQxH0y2kmuu8rcok2J",
    "kme0dYumI8c4RVsw5cUd0aT4oWuFwIWB3lARzb7NTxzlAs7xRWPerB0ZDMfkWS096LWHHweBc5p8NKnFL+3esLK5UGXL/235v/fb",
    "/7YPu91dp7vb7rYOO1v+7wvj/4qxuX5LNvAD9r+tg661/+3u7u/t0v7fa3e3/l+fl//rCv/nw/gW9qvQSLg+VBgc682GbGM57pPA",
    "XYw1cYFvp4jGJjYO1Ss/FrYrIU5RV0WCDYtKSLEvOV+iPTqXVOIuPOKzXqQPWaL+8uUr5kFA70EunovQDXGIkwwfGfC4sdpAq8Th",
    "HpxDkbpAMixQhG8ZPISWSYeefjisSAw4sKjNuZukEuUtCiGBtnYhEMhf6FDHvqdsryz7SL1gI2biC8UehSqhNRTNKkale65dagoH",
    "zqtUxOMs4TE0g3f856MnbxVrXnY0jzY1coxwOiFUPVAZ5VpNdc4G1YgwAX7IWyDyAga2wrPiqmQZsPj6k/nPYhyeTzS1vYcXffXm",
    "6fFLovXuMUzlz6C/PF49TSoT6rS5V7U0WBH9bOBji3a7RZ6Ww+OMeICzCDkrfK64LYbpNI7mNKWmG0c2oVLi8zZyw2XGzzC5K9zW",
    "hkAS1TAycOFuSCTe8lfR7oz98YqmKF6E/2JNZQobgPiJTBulx6P8i2FtROsPJO5NLjJGfcCheWxzxUaCPhY0QH+JFmy0TytoSoPB",
    "jnay04muJebiwocbKHJNNqAEiw38kJ0oaWsW1ECi85HN7GKUEEqDjYRdhBWhLe0GQAu5ZyLaKpUZqvXs7ekL2iT/7ezNa1Z5ZUog",
    "/AvvPOrMhBb8kdnWPXW7vtPvKhVGZ+IceZutkjuzG8HeEd4qDPa5Jq7AURxtksfi1Y9nb3uVJuJCnuv0WuuQKjKDPXhIDOnI7F6q",
    "jXFV8SvVV/hqfvJoEUREhxIvgmkUXcpIBS4yAEn4pXqQYzQjJDNCjpFk2ACbwJopUm9eH7OlBiItGVEXcI8rmIXQDqanhkVM3fcD",
    "vEWTSZ0g8AQpnqEeVgeG54IGnMtzwBfUQcydWiA2J2ensUxpsAmV6mzoZCaxOzLlsx0C0SXnxnrZAi9HH8ICZ0Wx0UOvFs+LHT17",
    "e3zKJQO3VDD100BXgRYScNk7qQRwafD2TRkZ1P6431IwqUikCAabSrCNDjurYKYwPtDMco7r6ZIyYAUnUK7i3KCJpFQaBXOWIAQp",
    "D9jb71+cKbskubQcjj3VbrZbqsaTDyEni72+yeYMDhf0akvmR5Q08tLnEamKlrvK7hx0wnpRMJbIXMT2LeJ5lOAbR2zSrGBl+co0",
    "uq7bk4m3GO2eW4MwemoAbw01vLszcjMI57jNLJRzYMc5jwkwTv/To9dnT05fnLxVNZ74gH18BjzfTZq0IUsBiJO9zTfZHQdSNVhL",
    "JCAZIjYs8QP1HIdvtKAeaV6cPbHJMoNKMBH8dSyEBRMG7BFkpA3nehIZdyRGIFTztTXrpnGcA0ty1Q6tnwRxax2iSiBQqZScPfp8",
    "YsnxIqD77X0iWBsGofblT27KYMH1B7fVOOKFVwWy4kMPmybELkDK3bBorvhAvZLezdyl0jM/zTt2DmlnqUu89TiZDxLIrcQqgD+Y",
    "ajID02DJEXwg5DIyLjqa3TSNawyCGobPLBURqZdAFguZuWPA5acMOykgyJcR1uRV1Mtnjs08GIGsofFjA56k5omJRSK1eaiJs66U",
    "8lY+48j1hpuFWw/UmFYPoaMmU1CghnJgDlZxrSDr9XrKM6YpvBMbqlWHhAeBfXXBZ982hf9yBLyRaegG1xepKnd9WfeysNtLTqD7",
    "vCzed9QXbZoaikNKWXqn3SlLpLAPiaD02fSpYSSkmWS0IBjNiRdvuggvpUdjK+rkOoYFu1za1og0BaE65x+0hpkNUcMkNdtDYyNU",
    "KYoCi2LWDbJALptLP7kD1vJpUh3cciU9pzO5a97CrwW/huoWkK1nSyaH5GYXB7j6U2hqZriZzG9lZdLMWIpybcCzAI/GSyyzOToj",
    "GFTPvy00x8vOyXqx8SFi3WUTHzpJFus0fRZNomS9wkBgKO2beSopK67pWD5PCtZbqqnEOAs9yG266vXsdy6a57n7EFyevRWoYvUF",
    "mPncGluZ2g96yeRuo+CWtC6+xLBZ6hntaJp+/lHt3p+31KWGKrQFlDobpplluYtdzAn0Ga8GtRKyw9GUaRQxpyLfB+8qFG+UaRad",
    "+VKNI83RCWPiUTjMXKElhgtgifaKJJsDPHCPmqqtm7tQQRQm6Y/c6W/4U0lCC8CZFDrHrFn4z5XYorRijm8IN3hpfsJF579oejfR",
    "WcRBgTEzEUkRe2jgLJnozMgqWcQ8Vqxh4dM8W4iyVzluaDFcKHXOpGURSqo///xz0Xa0VJTVGZJj0B46sZ5FV3pOpIh/U5NQiOVw",
    "pKUNsB6Wjk+Y4qLjb+j8Uw1jXV5zvc2oihsELqZWvYVpKr/HklBCHnb+/tBXzbaEOqT/vpPUlYgfG1solE/PTHRBUfThmKGFSVM5",
    "n0XsI09wzehFb4G6eSzuvjVU8m7LkK89QYeDHqUM/xDf1YvBQ7f2X1v5/0fI/1utw8NHTqez/+hwb28r///C5P/eIv09wr9/MP77",
    "3v5uFv+9K/HfO91t/PfPK//fY3byCS0BdRbAlp3N2kEegdUxghkTVktFTC25ARuj08kOOoKvXKE0NYYE3fX5Whi2Flfm0hQRmpgw",
    "w3T2w5zdSAv4rB46FXByIwsJLsBEvRAVBGoia9DDxBJcMKRHQA2i7+BI1RBG8MKdi5hNg8hzw8rPF6Dzp0SHT6Ng/HODj3Y6ZD0x",
    "/lJzX4MwEmMyUWiIaH/mj+Wmm+piNkuqO3OXmPfE0PLU9ys39JPpx4ve/2tR7mhbZiHujPi7zKrcIxYv9bon1DzsZZyDFfH4Ci9L",
    "S4Bj063zfPcS41kcuxXxszDamya1WpdZoHxFH5sCOZlpdfpqJJM8wiTXSoT4Si83RnnL4QxqeX/qeaNBVNlc9RXzKlpU8Kazn0sx",
    "tzeFFM+iWy8lkDgs5CbV26Tn7E7uOCmNJEnnSTYStolCtxrmzutxeHJaNDed/S6X8HouR/lwPX5l6xUW40DnhnAFGyPjFSnSguyh",
    "aKJlOtqT/clSPepnA/skNHvGUT9wDMGjnT+xl9Qy9GCmJdvIMdBOdVNL9PzIlhN5c9NTiGlXB9eVTQ2L9Qq3Oqnnb04SkW15gXah",
    "ATDxsGOZyNyhy2+oGiapXraNyuazEAk+X+STKiCNbv1eqzO+y5f77zmteaC/33KC5xvBFiaah8yKUuZGtsXzwfdjFDe+zG56k1bN",
    "JB6dE2i4kwJnJSxFH8vBwFy09X2S2R3r2eJGizaV2pPQjws3HgeQZNPx4V0bd/tC5cXQ5lY+Oalysx7eFnp29/CnsMqzzRZw3CXr",
    "SFqarfWZmhjhqydRJJu4cAM/WqUZKrZp03j+v+7BNaDD+ySARUy3Seznzg0mz53g5IYHoxThP8PcJ+5JFAR0HurikWmcDSFfL52e",
    "57F2WepMWzahoWBxwXeo0+peMjEBCpiW8T+m2vLW3GSS6k/E55ktxYsiLKSy8LA9hOQEHS1z3IXvLFgpyxjXbRNRwK77wXXh3LJl",
    "h1ZElMwQVmzujlkeQqv6WqtxBBEQH/4YFix/lw9lpccXOinO3qA2c29qUAtQN1pO6xBBnNQ38lOcTpHgs443Gf5tsVXb+O/b+O9F",
    "/v/R3p7T6nQeHXa37P+Xxv+bSzh/cxnAB/j/g71Wfv/bHnBBu0v5t/z/Z+X/95n/PTX3sLb3e49AYDzqtfeN77aJtATVAxMJNdB+",
    "U6LwmB5jgzeYCQXsM380n8cRMQ89vj/H6ComrqeFpZBKaq9wQSsCSyDSjw+3nilO3pkfusQDIHsWb0S5KkVQobSCOObNG+XB9iKW",
    "yFM+ET7JjCZI9DzIIUHe2TcI6E3V9M08CuHShYDvnJc6wVyIn6oLuKQklYL53y8+3NL5/jtmfQKdWskFu1uJ4EDHTYaOGqnX9i4l",
    "6F6Ncw9okjZbTfBttHDxzyO8qDenEPGjo4mEThXShXW5bup7ppfSo5obXLtLFn5csuFXx1HPVou+/wFg7rsec61Nnk3YeFIFlcpJ",
    "7ve3gCf/WCj/nkquiQZk2Yi5hBdUzfMogs9VNonq66+PFmn0jDDK119bc0kiqpJK4sLewlsSUURseto0CtCl4vCusHVxFJuSMulv",
    "7EkRXuFbGOqsxkDwprBISn4fscvqfWoNEcS8+fHt6F1D4Q8uGESk+4ZqP+q07h/rB7hROArZwi03dXPDKzdRRDGm3pRmzuod87ua",
    "eGuwkGN0b9jS97j0lWU5FqNnDC71Ae1fuzUur8+o43ivjeAdhSusRLH6vfpaPSKw7f16Xf2z+mu72FesUL5yTNUmiNMD89+Q14nh",
    "meqZvCUD/l1fveu9f8U+UG4Ai8xlfve0n9CK5Pp4Z8IBiNbXr7rMwa+zghmzl/d1JYJJtXk1YS4e0Pu30JPTgNnm1u96t9/fNRJq",
    "g+7f8oqgFF4Sd9U1SIY7hKyjIIt5jwRGtjqLu8yuHJmkQosbPIN2BZSuNDOZi/dDnK0jkYlBUQ3a4dyTzFIhX62ySDM4N7AcfUe8",
    "TTYQamdHdX7z4bbgeZh7tzd3vdbvNtgP1GNcMIhThTZB84qQEbAMD9FNk84KYtixM0SnnuFMO0cwLTenQUK8pDcbW7mXeTVhn2Fv",
    "GbCNHqO2kE893J7HpxNr+2PEjrnUHA36Ye+h0qlHp6EwoWA3cfga2N71uC92tGwVb6z6ePLO3YShN6yZT0iH5o65lo/OkiVsO1iW",
    "Y4QwM7Z1zkQ/OL8oTZDE1A0m0JvbbWonm61nyiw+HbzeDRvwrK4+rBrDF/M+KqwfLtPkaur32vzcirmPmRJ1e9Nz2pO7b42+Hu0v",
    "SY9WTHwKwqGPWZmbZXPZ+jST2p/0b7lijPRdY9OqbX3Mmr1nvRYaAek6TTY+YIrq99qLrSCK0onxjs3DCF2Yv2XtwGSejBIX6plc",
    "NdB1WvXcsok/CqlGxyCbgZdoMzPnRItB+iuwcP1NRmch8l1OlfFdhhvtnczR6111VpNymokOzdn84yL0DIxkn85WKC2uOg5fEvlE",
    "bpdcC0WXxN5oIqZ+rugiUOTJ0cno5PTNyejZyRmb1O62jEJEz83KbpubNU35ncKQGtBC+0awVp/NHd5/bGSE0RvJR0RTABH31L5J",
    "vL2R3EBK7/32+4PCvfeBhXpWj1xlPAbO6LecPdN7SwrDol7svYwKx9AOVomTZYTFpHO4V7dhPd3rHCX445vMI9sizZGHtHcZFhEE",
    "BUPLfOYiOo+EJ5BZwIm/fgFPdFmW8LG8spgJ1f+TmaLSnb285EDLUIYdO+MrBj3s428mzDGXlvJS8K7SJ1GASNsxY1heHm9evjkd",
    "PX5+2jl9/rheXw0KR9CcbNw3OOQ/yHdSX4LFJqk6j27WMuKKJLPcyjDfuygKVn1j2u8OrrniFQB/asceSaNzYwJGP24cpt3W/c/p",
    "E8LO41Kwj4CyHteOiQfAv6H+Ev2fVYX1UCeC8t1amfLCyRfo16Uv36haGzaEOSv3NdW2ci3TtT1PahmHyKWLIRVoSWTO67L2Ag0f",
    "8PIWdrwgyhKtb7V7/XcvJNvKf7fy32L8r/12y9ndbR90drfy3y/O/is3gv6c8t+9g72Wjf910Nrl+586B/tb+e9nlf8esPwX9zo1",
    "z5cScmf1aiYTAPXo7Az3urNvndz5Wam805mcZupe6VLYHhNK1TK17OAEGCxW/umny6weCfEj0btN+YoHyktN/YspXxhCnPOCJafM",
    "l5uoVJAwuUmiajlbXucQYfBLBHM6XkCSS8c2PBbhuCaxhBR8KuHbeXrcfHx0dvw0E4WwfVXefvGUtcYQxm/U2sKxwzkKLURfXBET",
    "eaiLWxzYDGOV2esbF1u2QkJHQbVawzmxjFLXNMwwmWJf8Gk2/PAOq4xjf5IighKNBbFXlMret+KshAnEnQzzFD5Qcvtn+1uGcapn",
    "0ZpcV2z3NI+SDXb2W0hVmTfFTS3jkZusOdZ8jBTTsG7WmMysP6PuXzFXs0iLqqqWHLlXbcs4fnPuL229bEpWbcLTTJkYgiSOgI7k",
    "rUYVmdIL5hVG7Gc0wpv1pqEsxjJtRAsYn6qmxg2iE/rKghMqv8H1B5ITp9XA56KXFVG8eePrq349VEgAwPigI4Vzf6x7yhq3jZHs",
    "DjS65uEOFupMY2OBkqBmxY3hKbH90QWxdqrVuB2libSnficvMLu7a7xy/bDRaPH/GsaFq+CqYKa4KN0x8/HNmieXSbpPPLI6Q2Vb",
    "GjNFLB3JrWnyHMNSHKlB0TPOFBUHNp9NO6CTgLArd67LaoCpoTSoPMbl5nhutkvyUS8a+2TufbRD34SaFxAsE690bDzLkm8Lt+DR",
    "yk8YvRL2k4Dr7Nid41IluJUw0ENiGFHmYSafKSDcXC5SzVJHXLS4pTZb5l2XjfIwFoVY8wsaiaQsTBFFQ9EJrRClnDi8dqtVr9dF",
    "GgqUZX3CUR/1tXShlpgJmdDoNlgWbFRtdzgN8b6q5h6FLHtBhv69psMjgRjdDFI+Nj1rZWzcZA3mIdiMc4GKQzPKTlFMwUskj8T3",
    "XV/t89zkg16WGMg89QvfB776JwaTJ9WHpTIls79J9faWRudWRvzup5+8W4Z5d3d3i1bc4TOlEfp6OI/9mRsvZYYfDimLqr4vtt/7",
    "KrLgVXmHWlM/sePL3MDMHimj3MLxkK3/8yjA5Dbbpdm85luoxZXZRneTS+POA9e75CurqSD+PkIIepnyVn5NBN+pJTLNak/tcf40",
    "jXA3TgeO0tGcfh3K9TdZrfMo8bG5WJwrZWnj75X6i0gVgzMJMfEinETDiry8Xc4JB1x1nVbrm8pJ4C5PdfLnHisX7etfeqxlrLyL",
    "3flZuoSwtlOpDP7U/UbxazJE2IWZS2vxNQunntFSDLNfcmfXiUwphFiLuKHeLFJgT/uKOzKz3zRAWYaGOpu64+i6oY7sZRMN9cqN",
    "iVd4aX+c2h9/aqhj2PsSXVUxDWU8L4tqQm2htdS4xR7nFLTsIW/p7l1j88pr/PP3ViRAPw/tz1vMoi0TSVMZtsDlJvM7T+pdY7+F",
    "/7c79B+N3PEVrILzUXvpLnH955lQZMdwHeXmN8x4vq+/k4kGtn5LRxRTTVYoT6eeoKWN2HtqdKoWde3sqN19QmvZrWv4mH39J/mI",
    "XPsmT8JY337eb5WX2u30rnc7Y7vm3m3Sa+1Bi1Kt/G3Lf7b+f//f5D8r8d/3d3edg87uQae7NQD88uz/mD/+zV0APyD/2c/8/yD/",
    "2ef47wfd3a3857PKfw5ZfPAmhlkSIoARzaoec3wCezG3mvk3mecfZCdEE58jHB9LdCaIYKtenXSJPcLNVqwkqlSOAuMlQuSTuaIr",
    "u+aGTuJ4qWo/G2Xbz3UJgyX+AUEAZvhCC23P0ZZY4KJYdwt6ucIcIozG4IVnAhVypLU3P77dMRZl7MsnQBHbwSO4nx6r/SPi42VK",
    "7IKszP7GhV+r2RZ8nRD92fCxGAgPucz7ajY/ZLWhm5hfG7IYHS7ymJ8bMhWuDaZ8+duGrBz6HbkkBnyeoWhCx9eqieymIc5jSbC4",
    "8CfLDaHrrZ7V+jdyuGUi6kZjPy5avo2w/WzC13JFX3YZNweDLytE10MCNmTFCoz33fUtLR8Zw8fVe8FzQQHgGBGBFIEqN+96rQCn",
    "nrtG8ruwHsXMmdgDHXdml/Qv2CO+302uxNU3VOkouiyEKPolOjeCMB4eaLZloK3HmYnCsVOtQwzEgcGoTDUr/TEVCXcfgyittncO",
    "zVr7z//4P9X6t5nUKvaYbrWr0JGfNSsiparqRTgdgpOvsxyWldyV12Dp6uDYWwe3S+BkZUqMfwswW6TO2g0JuQWdNaHDipJ7yrFM",
    "qqthHd97n5ZcpIpyO/b2CwbnwwLsqkOraHHO6JXwpBiGIUhoE56TQFuLNBUH6sz6uFpZvVVNe9MmjLxs7EVGClRK7upqNrk+bARG",
    "l2F07RRlasXR6tJosTGPwTB2tHi78N6EKYhBOc56UE4efzt+heCbK0HC2IQjh7kSHkzaMqmKrAjWyxzaRACZAI02GiafLFZ2x7tu",
    "Tc7EUvWyqIkhFeJb0UKnYtgycP9ksse4f1a/5a9mM2zYZ8UG79HoWUm/uvXvVI0DNQ4esjrk4XDQ67aGd3U7qGIdY9C8w57cWMEo",
    "0uBay6acxZr2qSaLv21t5W1XviQAGcccvzRD9I41J6ZmfKi6A3TMHl4frg/6ln7phHPKEv8P1XfI3QPBWa4tv4aRp91KlkbQ54gF",
    "DlEcVIkdRIMzCwsS+7BsMSegLH/OkEqWcgRuk2v9yjlUsp8rbND1c2VFmwGsDJ7dIGdROEhoymG9oNIw9fECvUXugmfy1YRDnlq9",
    "W9K/pUaLIaINpzjD7L/H3HHV0jGLZSsdyOWy5wXdC1/PWR0OBL2NiN4bjc8LAlLU+g2qbYp1/iiIojlX2y5VzsXvdYEWq0i58X0k",
    "MTBvqutZJtVBq3c1vL2a3A2uht8O2j13eBUFtOf7t+Pzu/HjwfkFJbcoGb9cIlj7bAmb9Ds9Kwbvc9ykgTusbmrDzOXWE3huvn2l",
    "3MMNsQXy3rOl6NVkWCknb3Abzv2EzdXb7Escs6NyZ/X25DUv43N5be+3LjdbjmaGr9SI3IoUi0WOeWu3Vw5RuLruOEJXo3g3uQ2l",
    "Wlq9jSxeaq5s44R6w8ZJLWjh6L3oU20joeY5JAVlTYDT/BMnlEozbTGysWVXYlPYzzoDYzQKBQhQXSOQK40fS0wr61eRFnRJfJ7N",
    "zQWkpZufmZzdHBQMmWvzOtsziiqrXt/6/27lf3/D8r9uy9nvHhx0uo+28r8v4BG5yO9bx4fuf2ztFeR/B4j/1abUrfzvczwP/rCz",
    "SOKdcz/c0eGVmi/TaRTuQkD2BFT5sdxp8eTlC+ZfIQdhgyMRwvkhgoa44ZiIvyWsssoh/floNbwnw7VSuGaTqTIiHtJ0nvR2dpaU",
    "c+Gc650XT6tgZt2bpkDYe0/pv+5MaE7cC72j53uglFGSpZJ/lYsad4hapfRdBDv/t4VPRFLPem0RtJOjt9831NHrt9+fvjl58WR0",
    "dPJi9MPxXxQRZ7i+nipn0VYsReVSnPQm/WThoRtfsKPXh0SIBUm8Yxij3CcVQjJDicxcP5TLr/NAxGwuZStyjuKLBdp7wv5lNcRt",
    "N7dM9KtnuMhhGnGwYrhsy8wJCyaVGxbMnTvumNg5A6tWNcNebdghGRuB0VQH8z4ulHi7ONfqx9OXEDTBWSAwywR9vh9qxDCpZ+4i",
    "SPvVNz++rVqYxn8NtCkY/bKg936IkIgVQZp3gUljAd/bDChb37GfUarfAxPylSJMkTmae9+T/sAmGPHR0FaXCbsZwP3g7YonEOly",
    "rvvssWVr27+/HNeW52Rxph29NVG9qhFHRH/cVLatanZa48f1+5slXOA98FmSmUaqcClqzeTrrV+YmtUSX0ByQJVJsGO81+qVkvDI",
    "SoVzszim5vvI6/AqzMl4w7z0mdTmDOWYZ5l4tJCDfeoapWj/Ahu/CulWtGU+2tdCDoxsAbDIHsBT568i7C3LmQuyYIFthBeFSwGs",
    "nOSn8CkVdTbIxyDwurU9vtvpmQEuB87vrQleFGHy/61uPSM8gN0N7gcd4ZUwGUSgoxGQzGhkpKCCcbbk4D/is+X/tvxfkf/rHu45",
    "+63dg/bewXbDfwn2H6vn9Ofn/7qdvbbl//b3d/dh/wGWcMv/fQ7+Tz2OTThkqLAlBrK296kbkwtEf44XRO2BtphG14b+CHB5FV8n",
    "AQ2eU3mgnnBonkwttDN3PWg2vz8+PZY7t0CWEI2VG2fwzZFygaL/K3NOVQkldc0BGC8QOoO4Q2IjLVDQJLAn7anqK5i46pgI+mrR",
    "t9q69eRxliJx2TFs1cz1pj5fMCI2wwTpMZsIr/hoP2q1sgFhm+Pvo3gW/eo35RI7XCZCI0BgYMzaU5uDAT1gX6dFiKgfNQ4NYcLd",
    "AipMfZvXCKqacc6s4gIlVzKHpSbCGvYZP1UCej3Ffek1wKZRxLfHj58/Pz1FyQfqh1VTdbEhFxehTquzL47lc7gl+bC7wFVsj0+P",
    "Xj81GWW2Qh2FDgN8EQQLOKyzH8FJHPF9I+7snL0ynvGdUWLu7qgzxIrCQqFpi+W+uam+AZhV9wGhLpvStVb3SffZQXUl9g/XoGoP",
    "nh086ba69WKBzu6zx63OSoGLGHf0UYFW59njzm6dO4zOJcRtYE0vK8wyyE1lu5umK50iEpnJI+E7vTgKgmagL/xzH3eBYsrZzrjH",
    "fuPWBryXmYCXokBJlAwCQk1r8CDwTYqxm6RUOHNCQMjlBS/KogPRSiMfZM5yuLHKDRL1111jNkFMnUSzqQjnomfRL76Bw0rNlgUh",
    "vF+8gJE2h1duqIm+3gkjJWUqFdb6YXqKWr8e2MVSW16cGN6SrwIU8wPrC0HNYe5S1aS6MAqbob6IUh/OafUMNt9mOmIAo1Tzzrww",
    "Q1Ex6ms0xIRZYRUejTPi0kExmrn/9zgoxWoALoSKM45veRyBWqsv0d2WDdXuT+LoVx3WHZRvtpxH2E60D2dYsLGWyRtF4Sihn5pm",
    "JLWNy9irwjWQtB4OzfBnKQeQYb33CkeaY8qyOfQ96uKucIRijasF4VDYzEJnp3wrrEKA+ixefYVQ8TPX5yAL8zymG9u/LAIEXite",
    "fIktxs5C13EEg5JnLCMwV85lJgHQmhNGBZYP9IQ98GBX947wOMK/4EY+bozIbBoqZrQDRXJMw0h44Qmt8SahZIQc44hIHAMNfk+0",
    "PVkpGl9pOkXEakcsefj6SFclECDRrB8l7nxa5bQkcSflpAljIRbDULrgpLd4w0e/iL/mgr+QbRNe4wLJZRQFyHHGP5B0wV6VhJY9",
    "vtnvOb+e8CvXH/FFX7RiQ2kBOk4JzzgBU2JvKzWhToBaxLKmaYx5Vq4gkGkqXSu8eisxX80IM6CKm12F+h1CA+41d/fUUruxiuT2",
    "VbZKkguPJfoy33mIeRj7MYe4QKw7TPpMj5v6Cgd5wCcSMAQCIQJzTlKEgTclgyUt2iWkfVgcOGZxe8QJ1+EGcoRMl3M67c/4Akkj",
    "FA4LFy6b6FZ8TSYto8TezbnaT2fLjGyf7bN9ts/22T7bZ/tsn+2zfbbP9tk+22f7bJ/ts322z/bZPttn+2yf7bN9ts/22T7bZ/ts",
    "fv4vDYgt1QDwAAA=",
]
blob = base64.b64decode("".join(BLOB_PARTS))
tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz").extractall(".")
print("\u2705 Engine loaded.")

### 3️⃣ Your settings — edit the two lines, then click ▶

- **VIDEO_URL** — paste a YouTube link, e.g. a Flock Talk episode
- **ANTHROPIC_API_KEY** — get one at [console.anthropic.com](https://console.anthropic.com) → API Keys → Create Key (starts with `sk-ant-`)

In [ ]:
VIDEO_URL = "https://youtu.be/PASTE_YOUR_VIDEO_LINK"  #@param {type:"string"}
ANTHROPIC_API_KEY = "sk-ant-PASTE_YOUR_KEY"           #@param {type:"string"}
MAX_CLIPS = 3                                          #@param {type:"integer"}
print("✅ Saved. Video:", VIDEO_URL, "| clips:", MAX_CLIPS, "— go to step 4.")

### 4️⃣ Make the clips. Click ▶ and let it cook (5–15 min; the first run also downloads the speech model).

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["CLIP_ENGINE_ASR"] = "faster_whisper"

from pathlib import Path
from clip_engine.render import process

clips = process(source=VIDEO_URL, out_dir=Path("OUT"), work_root=Path("work"),
                mode="talk", max_clips=int(MAX_CLIPS))
print(f"\n✅ Done — {len(clips)} clips made. Go to step 5 to download.")
for c in clips: print("   •", c.name)

### 5️⃣ Download your clips. Click ▶ — `clips.zip` saves to your computer.

In [ ]:
!zip -qr clips.zip OUT
from google.colab import files
files.download("clips.zip")
print("✅ If no download started: click the 📁 folder icon on the left, right-click clips.zip → Download.")

---
**Tweak the look:** open the 📁 folder icon on the left → `config` → double-click `brand.yaml` — fonts, highlight colors, clip length all live there. Re-run step 4 after editing.

**Something errored?** Copy the red text and paste it to Claude — it built this and will fix it.